# Feature Engineering

In [49]:
from pathlib import Path
import pandas as pd
import numpy as np

In [50]:
# PATH DEFINITIONS
BASE_DIR = Path().resolve().parent

DATA_DIR = BASE_DIR / "data"
CLEAN_DATA_DIR = DATA_DIR / "data_clean"

In [51]:
# NOT THE IDEAL, BUT IT WORKS, AFTER, SEARCH FOR ANOTHER APPROACH ON cleaning.py

df_clean = pd.read_csv(CLEAN_DATA_DIR / "train_clean.csv", keep_default_na=False, na_values=[""])

## Classification

In [52]:
df_clean.head()

,ID,MSSUBCLASS,MSZONING,LOTFRONTAGE,LOTAREA,STREET,ALLEY,LOTSHAPE,LANDCONTOUR,UTILITIES,...,FENCE,MISCFEATURE,MISCVAL,MOSOLD,YRSOLD,SALETYPE,SALECONDITION,SALEPRICE,HASGARAGE,HASPOOL
0,1,60,RL,65.0,8450.0,PAVE,NA,REG,LVL,ALLPUB,...,NA,NA,0,2,2008,WD,NORMAL,208500,True,False
1,2,20,RL,80.0,9600.0,PAVE,NA,REG,LVL,ALLPUB,...,NA,NA,0,5,2007,WD,NORMAL,181500,True,False
2,3,60,RL,68.0,11250.0,PAVE,NA,IR1,LVL,ALLPUB,...,NA,NA,0,9,2008,WD,NORMAL,223500,True,False
3,4,70,RL,60.0,9550.0,PAVE,NA,IR1,LVL,ALLPUB,...,NA,NA,0,2,2006,WD,ABNORML,140000,True,False
4,5,60,RL,84.0,14260.0,PAVE,NA,IR1,LVL,ALLPUB,...,NA,NA,0,12,2008,WD,NORMAL,250000,True,False


In [53]:
qual_nom = [
    "MSSUBCLASS", "MSZONING", "STREET", "ALLEY", "LANDCONTOUR",
    "LOTCONFIG", "NEIGHBORHOOD", "CONDITION1", "CONDITION2",
    "BLDGTYPE", "HOUSESTYLE", "ROOFSTYLE", "ROOFMATL",
    "EXTERIOR1ST", "EXTERIOR2ND", "MASVNRTYPE", "FOUNDATION",
    "HEATING", "CENTRALAIR", "GARAGETYPE", "MISCFEATURE",
    "SALETYPE", "SALECONDITION", "HASGARAGE", "HASPOOL"
]

qual_ord = [
    "LOTSHAPE", "UTILITIES", "LANDSLOPE", "OVERALLQUAL",
    "OVERALLCOND", "EXTERQUAL", "EXTERCOND", "BSMTQUAL",
    "BSMTCOND", "BSMTEXPOSURE", "BSMTFINTYPE1", "BSMTFINTYPE2",
    "HEATINGQC", "ELECTRICAL", "KITCHENQUAL", "FUNCTIONAL",
    "FIREPLACEQU", "GARAGEFINISH", "GARAGEQUAL", "GARAGECOND",
    "PAVEDDRIVE", "FENCE"
]

quant_disc = [
    "YEARBUILT", "YEARREMODADD", "BSMTFULLBATH", "BSMTHALFBATH",
    "FULLBATH", "HALFBATH", "BEDROOM", "KITCHEN", "TOTRMSABVGRD",
    "FIREPLACES", "GARAGECARS", "MOSOLD", "YRSOLD"
]

quant_cont = [
    "LOTFRONTAGE", "LOTAREA", "MASVNRAREA", "BSMTFINSF1",
    "BSMTFINSF2", "BSMTUNFSF", "TOTALBSMTSF", "1STFLRSF",
    "2NDFLRSF", "LOWQUALFINSF", "GRLIVAREA", "GARAGEAREA",
    "WOODDECKSF", "OPENPORCHSF", "ENCLOSEDPORCH", "3SSNPORCH",
    "SCREENPORCH", "MISCVAL"
]


In [54]:
df_qual_ord = df_clean.copy()

for col in qual_ord:
    print(df_qual_ord[col].value_counts().sort_index(ascending=False))
    print()

LOTSHAPE
REG    886
IR3     10
IR2     40
IR1    476
Name: count, dtype: int64

UTILITIES
NOSEWA       1
ALLPUB    1411
Name: count, dtype: int64

LANDSLOPE
SEV      13
MOD      64
GTL    1335
Name: count, dtype: int64

OVERALLQUAL
10     17
9      43
8     166
7     313
6     371
5     381
4     103
3      14
2       3
1       1
Name: count, dtype: int64

OVERALLCOND
9     22
8     72
7    200
6    248
5    792
4     52
3     20
2      5
1      1
Name: count, dtype: int64

EXTERQUAL
TA    871
GD    478
FA     12
EX     51
Name: count, dtype: int64

EXTERCOND
TA    1239
PO       1
GD     145
FA      24
EX       3
Name: count, dtype: int64

BSMTQUAL
TA    648
GD    609
FA     35
EX    120
Name: count, dtype: int64

BSMTCOND
TA    1301
PO       2
GD      64
FA      45
Name: count, dtype: int64

BSMTEXPOSURE
NO    944
MN    114
GD    133
AV    221
Name: count, dtype: int64

BSMTFINTYPE1
UNF    426
REC    132
LWQ     74
GLQ    412
BLQ    148
ALQ    220
Name: count, dtype: int64

BSMTFINTYP

In [ ]:
columns_to_drop = ["UTILITIES"]
columns_to_add = []

# GRADES 1-10
mapping_1_10_grades = {
    0: 1,
    1: 1,
    2: 1,
    3: 1,
    4: 1,
    5: 2,
    6: 2,
    7: 2,
    8: 3,
    9: 3,
    10: 3,
}
df_qual_ord["OVERALLCOND"] = df_qual_ord["OVERALLCOND"].map(mapping_1_10_grades)
df_qual_ord["OVERALLQUAL"] = df_qual_ord["OVERALLQUAL"].map(mapping_1_10_grades)

# FIVE QUALITIES
map_qual_5 = {
    "NA": 0,
    "PO": 0,
    "FA": 2,
    "TA": 2,
    "GD": 3,
    "EX": 3,
}
df_qual_ord["EXTERQUAL"] = df_qual_ord["EXTERQUAL"].map(map_qual_5)
df_qual_ord["EXTERCOND"] = df_qual_ord["EXTERCOND"].map(map_qual_5)
df_qual_ord["HEATINGQC"] = df_qual_ord["HEATINGQC"].map(map_qual_5)
df_qual_ord["KITCHENQUAL"] = df_qual_ord["KITCHENQUAL"].map(map_qual_5)

# SIX QUALITIES
map_qual_6 = {"NA": 0, "PO": 0, "FA": 2, "TA": 2, "GD": 3, "EX": 3}

df_qual_ord["BSMTQUAL"] = df_qual_ord["BSMTQUAL"].map(map_qual_6)
df_qual_ord["BSMTCOND"] = df_qual_ord["BSMTCOND"].map(map_qual_6)
df_qual_ord["FIREPLACEQU"] = df_qual_ord["FIREPLACEQU"].map(map_qual_6)
df_qual_ord["GARAGEQUAL"] = df_qual_ord["GARAGEQUAL"].map(map_qual_6)
df_qual_ord["GARAGECOND"] = df_qual_ord["GARAGECOND"].map(map_qual_6)

# BSMT TYPE
map_bsmt_fin = {
    "NA": 0,
    "UNF": 1,
    "LWQ": 2,
    "REC": 2,
    "BLQ": 2,
    "ALQ": 3,
    "GLQ": 3,
}
df_qual_ord["BSMTFINTYPE1"] = df_qual_ord["BSMTFINTYPE1"].map(map_bsmt_fin)
df_qual_ord["BSMTFINTYPE2"] = df_qual_ord["BSMTFINTYPE2"].map(map_bsmt_fin)

# SPECIAL CASES
df_qual_ord["LANDSLOPE"] = df_qual_ord["LANDSLOPE"].map(
    {"NA": 0, "SEV": 3, "MOD": 2, "GTL": 1}
)
df_qual_ord["BSMTEXPOSURE"] = df_qual_ord["BSMTEXPOSURE"].map(
    {"NA": 0, "NO": 1, "MN": 1, "AV": 1, "GD": 2}
)
df_qual_ord["GARAGEFINISH"] = df_qual_ord["GARAGEFINISH"].map(
    {"NA": 0, "UNF": 1, "RFN": 2, "FIN": 3}
)

df_qual_ord["ELECTRICAL"] = df_qual_ord["ELECTRICAL"].map(
    {"MIX": 1, "FUSEP": 1, "FUSEF": 2, "FUSEA": 2, "SBRKR": 3}
)

df_qual_ord["FENCE"] = df_qual_ord["FENCE"].map(
    {"NA": 0, "MNWW": 1, "GDWO": 1, "MNPRV": 1, "GDPRV": 1}
)

df_qual_ord["FUNCTIONAL"] = df_qual_ord["FUNCTIONAL"].map(
    {
        "SEV": 0,
        "MAJ2": 0,
        "MAJ1": 0,
        "MOD": 1,
        "MIN2": 1,
        "MIN1": 1,
        "TYP": 2,
        "NA": 0,
    }
)

# BOOLEAN CONVERSION
df_qual_ord = df_qual_ord.rename(columns={"FENCE": "HASFENCE"})
df_qual_ord["HASFENCE"] = df_qual_ord["HASFENCE"].astype("bool")
columns_to_drop.append("FENCE")
columns_to_add.append("HASFENCE")

df_qual_ord["PAVEDDRIVE"] = df_qual_ord["PAVEDDRIVE"].map(
    {"NA": 0, "MNWW": 1, "GDWO": 1, "MNPRV": 1, "GDPRV": 1}
)
df_qual_ord = df_qual_ord.rename(columns={"PAVEDDRIVE": "HASPAVEDDRIVE"})
df_qual_ord["HASPAVEDDRIVE"] = df_qual_ord["HASPAVEDDRIVE"].astype("bool")
columns_to_drop.append("PAVEDDRIVE")
columns_to_add.append("HASPAVEDDRIVE")


df_qual_ord["LOTSHAPE"] = df_qual_ord["LOTSHAPE"].map(
    {"REG": 1, "IR1": 0, "IR2": 0, "IR3": 0}
)
df_qual_ord = df_qual_ord.rename(columns={"LOTSHAPE": "HASREGULARLOTSHAPE"})
df_qual_ord["HASREGULARLOTSHAPE"] = df_qual_ord["HASREGULARLOTSHAPE"].astype("bool")
columns_to_drop.append("LOTSHAPE")
columns_to_add.append("HASREGULARLOTSHAPE")

# UPDATE QUAL_ORD
columns_to_drop
qual_ord = list((set(qual_ord) - set(columns_to_drop)))
qual_ord = list(set(qual_ord).union(set(columns_to_add)))


In [56]:
for col in qual_ord:
    print(df_qual_ord[col].value_counts().sort_index(ascending=False))
    print()

LANDSLOPE
3      13
2      64
1    1335
Name: count, dtype: int64

HASPAVEDDRIVE
True    1412
Name: count, dtype: int64

OVERALLQUAL
3     226
2    1065
1     121
Name: count, dtype: int64

EXTERCOND
3     148
2    1263
0       1
Name: count, dtype: int64

FUNCTIONAL
2    1322
1      71
0      19
Name: count, dtype: int64

HEATINGQC
3    959
2    452
0      1
Name: count, dtype: int64

HASFENCE
True      277
False    1135
Name: count, dtype: int64

EXTERQUAL
3    529
2    883
Name: count, dtype: int64

FIREPLACEQU
3    396
2    340
0    676
Name: count, dtype: int64

KITCHENQUAL
3    672
2    740
Name: count, dtype: int64

BSMTFINTYPE2
3      33
2     133
1    1246
Name: count, dtype: int64

OVERALLCOND
3      94
2    1240
1      78
Name: count, dtype: int64

GARAGEQUAL
3      17
2    1318
0      77
Name: count, dtype: int64

ELECTRICAL
3    1300
2     109
1       3
Name: count, dtype: int64

HASREGULARLOTSHAPE
True     886
False    526
Name: count, dtype: int64

GARAGEFINISH
3    345


In [58]:
pd.set_option('display.max_columns', None)
df_qual_ord.head()

,ID,MSSUBCLASS,MSZONING,LOTFRONTAGE,LOTAREA,STREET,ALLEY,HASREGULARLOTSHAPE,LANDCONTOUR,UTILITIES,LOTCONFIG,LANDSLOPE,NEIGHBORHOOD,CONDITION1,CONDITION2,BLDGTYPE,HOUSESTYLE,OVERALLQUAL,OVERALLCOND,YEARBUILT,YEARREMODADD,ROOFSTYLE,ROOFMATL,EXTERIOR1ST,EXTERIOR2ND,MASVNRTYPE,MASVNRAREA,EXTERQUAL,EXTERCOND,FOUNDATION,BSMTQUAL,BSMTCOND,BSMTEXPOSURE,BSMTFINTYPE1,BSMTFINSF1,BSMTFINTYPE2,BSMTFINSF2,BSMTUNFSF,TOTALBSMTSF,HEATING,HEATINGQC,CENTRALAIR,ELECTRICAL,1STFLRSF,2NDFLRSF,LOWQUALFINSF,GRLIVAREA,BSMTFULLBATH,BSMTHALFBATH,FULLBATH,HALFBATH,BEDROOMABVGR,KITCHENABVGR,KITCHENQUAL,TOTRMSABVGRD,FUNCTIONAL,FIREPLACES,FIREPLACEQU,GARAGETYPE,GARAGEFINISH,GARAGECARS,GARAGEAREA,GARAGEQUAL,GARAGECOND,HASPAVEDDRIVE,WOODDECKSF,OPENPORCHSF,ENCLOSEDPORCH,3SSNPORCH,SCREENPORCH,HASFENCE,MISCFEATURE,MISCVAL,MOSOLD,YRSOLD,SALETYPE,SALECONDITION,SALEPRICE,HASGARAGE,HASPOOL
0,1,60,RL,65.0,8450.0,PAVE,NA,True,LVL,ALLPUB,INSIDE,1,COLLGCR,NORM,NORM,1FAM,2STORY,2,2,2003,2003,GABLE,COMPSHG,VINYLSD,VINYLSD,BRKFACE,196.0,3,2,PCONC,3,2,1,3,706.0,1,0.0,150.0,856.0,GASA,3,Y,3,856.0,854.0,0.0,1710.0,1,0,2,1,3,1,3,8,2,0,0,ATTCHD,2,2,548.0,2,2,True,0.0,61.0,0.0,0.0,0.0,False,NA,0,2,2008,WD,NORMAL,208500,True,False
1,2,20,RL,80.0,9600.0,PAVE,NA,True,LVL,ALLPUB,FR2,1,VEENKER,FEEDR,NORM,1FAM,1STORY,2,3,1976,1976,GABLE,COMPSHG,METALSD,METALSD,None,0.0,2,2,CBLOCK,3,2,2,3,978.0,1,0.0,284.0,1262.0,GASA,3,Y,3,1262.0,0.0,0.0,1262.0,0,1,2,0,3,1,2,6,2,1,2,ATTCHD,2,2,460.0,2,2,True,298.0,0.0,0.0,0.0,0.0,False,NA,0,5,2007,WD,NORMAL,181500,True,False
2,3,60,RL,68.0,11250.0,PAVE,NA,False,LVL,ALLPUB,INSIDE,1,COLLGCR,NORM,NORM,1FAM,2STORY,2,2,2001,2002,GABLE,COMPSHG,VINYLSD,VINYLSD,BRKFACE,162.0,3,2,PCONC,3,2,1,3,486.0,1,0.0,434.0,920.0,GASA,3,Y,3,920.0,866.0,0.0,1786.0,1,0,2,1,3,1,3,6,2,1,2,ATTCHD,2,2,608.0,2,2,True,0.0,42.0,0.0,0.0,0.0,False,NA,0,9,2008,WD,NORMAL,223500,True,False
3,4,70,RL,60.0,9550.0,PAVE,NA,False,LVL,ALLPUB,CORNER,1,CRAWFOR,NORM,NORM,1FAM,2STORY,2,2,1915,1970,GABLE,COMPSHG,WD SDNG,WD SHNG,None,0.0,2,2,BRKTIL,2,3,1,3,216.0,1,0.0,540.0,756.0,GASA,3,Y,3,961.0,756.0,0.0,1717.0,1,0,1,0,3,1,3,7,2,1,3,DETCHD,1,3,642.0,2,2,True,0.0,35.0,272.0,0.0,0.0,False,NA,0,2,2006,WD,ABNORML,140000,True,False
4,5,60,RL,84.0,14260.0,PAVE,NA,False,LVL,ALLPUB,FR2,1,NORIDGE,NORM,NORM,1FAM,2STORY,3,2,2000,2000,GABLE,COMPSHG,VINYLSD,VINYLSD,BRKFACE,350.0,3,2,PCONC,3,2,1,3,655.0,1,0.0,490.0,1145.0,GASA,3,Y,3,1145.0,1053.0,0.0,2198.0,1,0,2,1,4,1,3,9,2,1,2,ATTCHD,2,3,836.0,2,2,True,192.0,84.0,0.0,0.0,0.0,False,NA,0,12,2008,WD,NORMAL,250000,True,False
